In [ ]:
import os
import subprocess
from pathlib import Path

# Buscar dónde se descargó el dataset de wheel de zarr en /kaggle/input/
input_path = Path("/kaggle/input")
wheel_files = list(input_path.glob("**/*.whl"))

if wheel_files:
    # Instalar desde el archivo .whl local sin usar red (--no-index)
    wheel_dir = wheel_files[0].parent
    print(f"Instalando zarr offline desde: {wheel_dir}")
    subprocess.run(
        [
            "pip",
            "install",
            "--no-index",
            "--find-links",
            str(wheel_dir),
            "zarr",
        ],
        check=True,
    )
else:
    print(
        "⚠️ No se encontraron archivos .whl. Asegúrate de haber añadido el dataset de zarr a Input."
    )

import zarr  # Ahora se importará correctamente sin errores

print("✅ Zarr importado exitosamente de forma offline.")

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import blob_log  # Laplacian of Gaussian para blobs 3D
from skimage.filters import gaussian

import imageio.v2 as imageio

In [ ]:
# Ruta estándar para la competición
DATA_DIR = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")

# Comprobar qué archivos y carpetas hay disponibles
for path in DATA_DIR.iterdir():
    print(path.name)

## 1. EDA

In [ ]:
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

# Obtener las rutas de todos los volúmenes .zarr de entrenamiento
train_zarrs = sorted(list(TRAIN_DIR.glob("*.zarr")))
test_zarrs = sorted(list(TEST_DIR.glob("*.zarr")))

print(f"Número de muestras en Train: {len(train_zarrs)}")
print(f"Número de muestras en Test: {len(test_zarrs)}")

# Ver el nombre de las primeras 3 muestras de train
for zarr_path in train_zarrs[:3]:
    print("Muestra train:", zarr_path.name)

In [ ]:
# Cargar la primera muestra de entrenamiento
sample_path = train_zarrs[0]

# Abrir el volumen de imagen 3D+t
img_data = zarr.open(sample_path, mode="r")["0"]
print(f"Dataset cargado: {sample_path.name}")
print(f"Dimensiones (T, Z, Y, X): {img_data.shape}")

# Leer el grafo de anotaciones (.geff) correspondiente
geff_path = TRAIN_DIR / f"{sample_path.stem}.geff"
geff_data = zarr.open(geff_path, mode="r")

# Extraer coordenadas de los nodos
nodes_df = pd.DataFrame(
    {
        "node_id": geff_data["nodes/ids"][:],
        "t": geff_data["nodes/props/t/values"][:],
        "z": geff_data["nodes/props/z/values"][:],
        "y": geff_data["nodes/props/y/values"][:],
        "x": geff_data["nodes/props/x/values"][:],
    }
)

print(
    f"Total de centroides anotados en esta muestra: {len(nodes_df)}"
)

In [ ]:
# Tomamos el volumen en el tiempo t = 0 (primer fotograma)
# Dimensiones de frame_t0: (64, 256, 256) -> (Z, Y, X)
frame_t0 = img_data[0]

# 1. Proyección de Máxima Intensidad (MIP) en Z: colapsa la profundidad para ver todas las células
mip_z = np.max(frame_t0, axis=0)

# 2. Corte en el plano medio de profundidad Z (Z = 32)
slice_z_mid = frame_t0[frame_t0.shape[0] // 2]

# 3. Filtrar los centroides anotados que caen exactamente en t = 0
nodes_t0 = nodes_df[nodes_df["t"] == 0]

# --- GRAFICAR LAS VISTAS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Proyección MIP + Centroides anotados
im1 = axes[0].imshow(mip_z, cmap="magma")
axes[0].scatter(
    nodes_t0["x"],
    nodes_t0["y"],
    c="cyan",
    s=40,
    marker="o",
    edgecolors="black",
    label="Centroides GT (t=0)",
)
axes[0].set_title(
    f"Proyección MIP 2D (T=0)\n{len(nodes_t0)} células anotadas en t=0"
)
axes[0].set_xlabel("X (voxels)")
axes[0].set_ylabel("Y (voxels)")
axes[0].legend(loc="upper right")
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

# Subplot 2: Corte central Z
im2 = axes[1].imshow(slice_z_mid, cmap="gray")
axes[1].set_title(f"Corte Plano Z medio (Z={frame_t0.shape[0] // 2}, T=0)")
axes[1].set_xlabel("X (voxels)")
axes[1].set_ylabel("Y (voxels)")
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 2. Baseline

In [ ]:
OUTPUT_SUBMISSION_PATH = Path("submission.csv")

# Parámetros para la detección y seguimiento física (en voxels)
MAX_DISTANCE_VOXELS = (
    15.0  # Distancia máxima entre t y t+1 para considerar la misma célula
)
MIN_SIGMA = 1.5  # Tamaño mínimo estimado de célula (3D)
MAX_SIGMA = 4.0  # Tamaño máximo estimado de célula (3D)
THRESHOLD = 0.05  # Umbral de sensibilidad para detectar centroides


def detect_cells_3d(volume_3d):
    """Detecta centroides (z, y, x) en un volumen 3D en un instante t determinado."""
    # Filtro suavizado inicial para reducir el ruido
    vol_smooth = gaussian(volume_3d, sigma=1.0, preserve_range=True)

    # Normalización min-max
    v_min, v_max = vol_smooth.min(), vol_smooth.max()
    if v_max > v_min:
        vol_smooth = (vol_smooth - v_min) / (v_max - v_min)

    # Detección de blobs usando Laplaciano de Gaussiana (3D)
    blobs = blob_log(
        vol_smooth,
        min_sigma=MIN_SIGMA,
        max_sigma=MAX_SIGMA,
        num_sigma=3,
        threshold=THRESHOLD,
    )

    if len(blobs) == 0:
        return np.empty((0, 3), dtype=int)

    # Retorna las coordenadas enteras (z, y, x)
    centroids = np.round(blobs[:, :3]).astype(int)
    return centroids


def process_dataset(dataset_name, zarr_path):
    """Procesa una secuencia 3D+t completa: detecta nodos y conecta aristas."""
    img_group = zarr.open(zarr_path, mode="r")["0"]
    num_timesteps = img_group.shape[0]

    all_nodes = []  # Almacena dicts con datos de los nodos
    all_edges = []  # Almacena dicts con datos de las aristas

    global_node_id = 1
    prev_frame_nodes = (
        []
    )  # Almacena tuplas: (node_id_global, array_coordenadas)

    print(f"  Procesando {dataset_name} ({num_timesteps} frames)...")

    for t in range(num_timesteps):
        volume_t = img_group[t]
        centroids = detect_cells_3d(volume_t)

        current_frame_nodes = []

        # 1. Registrar Nodos del fotograma t
        for z, y, x in centroids:
            node_info = {
                "dataset": dataset_name,
                "row_type": "node",
                "node_id": global_node_id,
                "t": t,
                "z": int(z),
                "y": int(y),
                "x": int(x),
                "source_id": -1,
                "target_id": -1,
            }
            all_nodes.append(node_info)
            current_frame_nodes.append((global_node_id, np.array([z, y, x])))
            global_node_id += 1

        # 2. Conectar Aristas (Tracking) con el fotograma anterior t-1
        if prev_frame_nodes and current_frame_nodes:
            prev_ids, prev_coords = zip(*prev_frame_nodes)
            curr_ids, curr_coords = zip(*current_frame_nodes)

            # Matriz de distancias euclídeas en espacio de voxels
            dist_matrix = cdist(np.array(prev_coords), np.array(curr_coords))

            # Asignación Óptima (Algoritmo Húngaro)
            row_ind, col_ind = linear_sum_assignment(dist_matrix)

            for r, c in zip(row_ind, col_ind):
                if dist_matrix[r, c] <= MAX_DISTANCE_VOXELS:
                    src_id = prev_ids[r]
                    tgt_id = curr_ids[c]

                    edge_info = {
                        "dataset": dataset_name,
                        "row_type": "edge",
                        "node_id": -1,
                        "t": -1,
                        "z": -1,
                        "y": -1,
                        "x": -1,
                        "source_id": src_id,
                        "target_id": tgt_id,
                    }
                    all_edges.append(edge_info)

        prev_frame_nodes = current_frame_nodes

    return all_nodes, all_edges


# --- EJECUCIÓN PRINCIPAL Y SUBMISSION ---
test_zarr_paths = sorted(list(TEST_DIR.glob("*.zarr")))
full_rows = []
row_index = 0

print(
    f"Iniciando inferencia en {len(test_zarr_paths)} datasets del conjunto Test..."
)

for zarr_path in test_zarr_paths:
    dataset_name = zarr_path.stem
    nodes, edges = process_dataset(dataset_name, zarr_path)

    # Combinar primero nodos y luego aristas
    dataset_records = nodes + edges

    for item in dataset_records:
        item["id"] = row_index
        full_rows.append(item)
        row_index += 1

# Convertir a DataFrame y ordenar columnas según el formato oficial
df_sub = pd.DataFrame(full_rows)
columns_order = [
    "id",
    "dataset",
    "row_type",
    "node_id",
    "t",
    "z",
    "y",
    "x",
    "source_id",
    "target_id",
]
df_sub = df_sub[columns_order]

# Guardar submission final
df_sub.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
print(
    f"\n✅ Archivo {OUTPUT_SUBMISSION_PATH} generado correctamente con {len(df_sub)} filas."
)